# Лабораторная работа №1

## Компьютерная лексикография, терминологические базы (TBX) и Translation Memory (TMX) в Smartcat

**ФИО студента:** _______________________
**Группа:** _______________________
**Вариант:** ______ (предметная область: _______________________)

---

**Цель:** научиться программно обрабатывать стандарты обмена терминологическими
данными (TBX) и памятью переводов (TMX), реализовать поиск нечётких совпадений
(Fuzzy Match) и подстановку терминов глоссария, экспортировать результат в
CAT-систему Smartcat.

**Порядок выполнения:**

1. Выполните ячейки Блока 1 (установка библиотек) и Блока 2 (данные).
2. Реализуйте четыре функции в Блоке 4 вместо маркеров `# TODO`.
3. Запустите Блок 5 — все `assert` должны пройти без ошибок.
4. Соберите файлы своего варианта, экспортируйте CSV, оформите отчёт.

**Оценивание (10 баллов):** парсинг TBX — 2, парсинг TMX — 2, Fuzzy Match — 3,
интеграция глоссария — 2, оформление — 1.

## Блок 1. Setup — установка библиотек

In [ ]:
!pip install -q Levenshtein pandas lxml

In [ ]:
from __future__ import annotations

import os
import re
import unicodedata
from typing import Any, Dict, List, Optional, Sequence, Tuple

import pandas as pd

try:
    from lxml import etree as ET

    LXML_AVAILABLE = True
except ImportError:
    import xml.etree.ElementTree as ET

    LXML_AVAILABLE = False

try:
    import Levenshtein as _lev

    def edit_distance(a: str, b: str) -> int:
        """Расстояние Левенштейна между двумя строками."""
        return _lev.distance(a, b)

except ImportError:

    def edit_distance(a: str, b: str) -> int:
        """Расстояние Левенштейна (резервная реализация на чистом Python)."""
        if a == b:
            return 0
        if not a:
            return len(b)
        if not b:
            return len(a)
        previous = list(range(len(b) + 1))
        for i, char_a in enumerate(a, start=1):
            current = [i]
            for j, char_b in enumerate(b, start=1):
                current.append(min(previous[j] + 1,
                                   current[j - 1] + 1,
                                   previous[j - 1] + (char_a != char_b)))
            previous = current
        return previous[-1]


# ВАЖНО: атрибут xml:lang в парсере раскрывается в полное имя с пространством имён
XML_LANG = "{http://www.w3.org/XML/1998/namespace}lang"

pd.set_option("display.max_colwidth", 60)
print("Окружение готово. XML backend:", "lxml" if LXML_AVAILABLE else "ElementTree")

## Блок 2. Теория: структура стандартов

### 2.1. TBX (TermBase eXchange, ISO 30042)

Формат обмена терминологическими базами. Организован **по понятиям**
(concept-oriented): одна `termEntry` = одно понятие, внутри — по одной `langSet`
на каждый язык.

```xml
<martif type="TBX" xml:lang="en">
  <martifHeader>…</martifHeader>
  <text><body>
    <termEntry id="tid-001">
      <descrip type="subjectField">Предметная область</descrip>
      <langSet xml:lang="en">
        <tig>
          <term>flight controller</term>
          <termNote type="partOfSpeech">noun</termNote>
        </tig>
      </langSet>
      <langSet xml:lang="ru">
        <ntig><termGrp>
          <term>полётный контроллер</term>
        </termGrp></ntig>
      </langSet>
    </termEntry>
  </body></text>
</martif>
```

| Тег | Назначение |
|-----|------------|
| `martif` | корень документа TBX-Core v2 |
| `termEntry` | терминологическая статья (одно понятие) |
| `langSet` | языковая секция, язык задаётся атрибутом `xml:lang` |
| `tig` / `ntig` | компактная / расширенная группа термина |
| `term` | сам термин |
| `termNote` | грамматическая или иная помета (`partOfSpeech`, `usage`) |
| `descrip` | описание: `definition`, `subjectField`, `context` |

### 2.2. TMX (Translation Memory eXchange, версия 1.4b)

Формат обмена памятью переводов. Организован **по сегментам**.

```xml
<tmx version="1.4">
  <header srclang="en" segtype="sentence" datatype="plaintext"
          creationtool="…" adminlang="en" o-tmf="TMX"/>
  <body>
    <tu tuid="1">
      <prop type="domain">Aerospace</prop>
      <tuv xml:lang="en"><seg>The autopilot holds the altitude.</seg></tuv>
      <tuv xml:lang="ru"><seg>Автопилот выдерживает высоту.</seg></tuv>
    </tu>
  </body>
</tmx>
```

| Тег | Назначение |
|-----|------------|
| `header` | метаданные памяти: инструмент, язык оригинала, тип сегментации |
| `tu` | translation unit — единица перевода |
| `tuv` | translation unit variant — вариант на одном языке (`xml:lang`) |
| `seg` | текст сегмента |
| `prop` | произвольное свойство (`domain`, `client`, `project`) |

### 2.3. Типы совпадений в CAT-системах

| Совпадение | Диапазон | Действие переводчика |
|------------|----------|----------------------|
| ICE (101 %) | 100 % + совпал контекст | подстановка без правки |
| Exact | 100 % | проверить контекст |
| High Fuzzy | 95–99 % | косметическая правка |
| Fuzzy | 75–94 % | редактирование |
| No Match | < 75 % | перевод с нуля |

## Блок 3. Данные

Ячейка ниже создаёт **демонстрационные** файлы `demo.tbx` и `demo.tmx` — они
нужны только для автотестов из Блока 5. Файлы своего варианта
(`variant_NN.tbx` — не менее 15 терминов, `variant_NN.tmx` — не менее 10
предложений) вы создаёте самостоятельно: либо аналогичным кодом, либо выгрузкой
из Smartcat, либо вручную.

In [ ]:
DEMO_TBX = """<?xml version="1.0" encoding="UTF-8"?>
<martif type="TBX" xml:lang="en">
  <martifHeader><fileDesc><titleStmt><title>Demo glossary</title></titleStmt>
  </fileDesc></martifHeader>
  <text><body>
    <termEntry id="tid-001">
      <descrip type="subjectField">Demo</descrip>
      <langSet xml:lang="en">
        <tig><term>flight controller</term>
        <termNote type="partOfSpeech">noun</termNote>
        <descrip type="definition">Onboard flight computer.</descrip></tig>
      </langSet>
      <langSet xml:lang="ru">
        <ntig><termGrp><term>полётный контроллер</term></termGrp></ntig>
      </langSet>
    </termEntry>
    <termEntry id="tid-002">
      <descrip type="subjectField">Demo</descrip>
      <langSet xml:lang="en"><tig><term>payload</term>
        <termNote type="partOfSpeech">noun</termNote></tig></langSet>
      <langSet xml:lang="ru"><ntig><termGrp><term>целевая нагрузка</term>
        </termGrp></ntig></langSet>
    </termEntry>
    <termEntry id="tid-003">
      <descrip type="subjectField">Demo</descrip>
      <langSet xml:lang="en"><tig><term>telemetry link</term></tig></langSet>
      <langSet xml:lang="ru"><ntig><termGrp><term>канал телеметрии</term>
        </termGrp></ntig></langSet>
    </termEntry>
  </body></text>
</martif>
"""

DEMO_TMX = """<?xml version="1.0" encoding="UTF-8"?>
<tmx version="1.4">
  <header creationtool="demo" creationtoolversion="1.0" segtype="sentence"
          o-tmf="TMX" adminlang="en" srclang="en" datatype="plaintext"/>
  <body>
    <tu tuid="1">
      <prop type="domain">Demo</prop>
      <tuv xml:lang="en"><seg>The flight controller stabilises the aircraft.</seg></tuv>
      <tuv xml:lang="ru"><seg>Полётный контроллер стабилизирует аппарат.</seg></tuv>
    </tu>
    <tu tuid="2">
      <prop type="domain">Demo</prop>
      <tuv xml:lang="en"><seg>The operator checks the telemetry link.</seg></tuv>
      <tuv xml:lang="ru"><seg>Оператор проверяет канал телеметрии.</seg></tuv>
    </tu>
    <tu tuid="3">
      <prop type="domain">Demo</prop>
      <tuv xml:lang="en"><seg>The payload is mounted under the fuselage.</seg></tuv>
      <tuv xml:lang="ru"><seg>Целевая нагрузка устанавливается под фюзеляжем.</seg></tuv>
    </tu>
  </body>
</tmx>
"""

with open("demo.tbx", "w", encoding="utf-8") as handler:
    handler.write(DEMO_TBX)
with open("demo.tmx", "w", encoding="utf-8") as handler:
    handler.write(DEMO_TMX)

print("Созданы файлы:", [f for f in os.listdir(".") if f.endswith((".tbx", ".tmx"))])

## Блок 4. Задание — реализуйте четыре функции

Требования ко всем функциям: типизация из `typing`, соответствие PEP 8,
обработка отсутствующих элементов без падения (`None`-проверки).

**Подсказки:**

* язык секции читается как `element.get(XML_LANG)`, а не `element.get("lang")`;
* обход `<term>` внутри `langSet` удобно делать через `lang_set.iter("term")` —
  это покрывает и `tig`, и `ntig/termGrp`;
* текст сегмента лучше собирать через `"".join(seg.itertext())` — так не
  теряется текст при наличии инлайновых тегов;
* при подстановке терминов сортируйте их по убыванию длины.

In [ ]:
def parse_tbx(file_path: str, src_lang: str = "en", tgt_lang: str = "ru") -> pd.DataFrame:
    """Разобрать TBX-глоссарий в плоскую таблицу терминологических пар.

    Args:
        file_path: путь к ``.tbx``-файлу.
        src_lang: код языка исходных терминов (значение ``xml:lang``).
        tgt_lang: код языка переводных эквивалентов.

    Returns:
        ``pd.DataFrame`` с колонками:
        ``entry_id`` (str)      — значение атрибута ``id`` у ``termEntry``;
        ``source_term`` (str)   — термин на языке ``src_lang``;
        ``target_term`` (str)   — термин на языке ``tgt_lang``;
        ``pos`` (str)           — часть речи из ``termNote[@type='partOfSpeech']``;
        ``definition`` (str)    — текст ``descrip[@type='definition']`` или "";
        ``subject_field`` (str) — текст ``descrip[@type='subjectField']`` или "".
        Статьи, в которых нет одного из двух языков, пропускаются.

    Raises:
        FileNotFoundError: если файл не существует.
    """
    # TODO: Студент пишет код здесь
    raise NotImplementedError


def parse_tmx(file_path: str, src_lang: str = "en", tgt_lang: str = "ru") -> pd.DataFrame:
    """Разобрать TMX-память переводов в таблицу параллельных сегментов.

    Args:
        file_path: путь к ``.tmx``-файлу.
        src_lang: код исходного языка.
        tgt_lang: код языка перевода.

    Returns:
        ``pd.DataFrame`` с колонками:
        ``tuid`` (str)            — идентификатор единицы перевода;
        ``source_segment`` (str)  — текст ``seg`` на языке ``src_lang``;
        ``target_segment`` (str)  — текст ``seg`` на языке ``tgt_lang``;
        ``domain`` (str)          — текст ``prop[@type='domain']`` или "";
        ``src_len`` (int)         — длина исходного сегмента в символах;
        ``prev_source`` (str)     — предыдущий исходный сегмент (для ICE), для
        первой строки — пустая строка.

    Raises:
        FileNotFoundError: если файл не существует.
    """
    # TODO: Студент пишет код здесь
    raise NotImplementedError


def calculate_fuzzy_match(source_query: str, tm_df: pd.DataFrame,
                          threshold: float = 70.0,
                          context_before: Optional[str] = None,
                          top_n: int = 5) -> List[Dict[str, Any]]:
    """Найти в памяти переводов сегменты, похожие на запрос.

    Алгоритм: нормализовать строки (нижний регистр, без пунктуации), посчитать
    процент сходства по Левенштейну и Token Sort Ratio, взять максимум,
    отбросить кандидатов ниже ``threshold``, отсортировать по убыванию.

    Args:
        source_query: сегмент, который нужно перевести.
        tm_df: таблица памяти переводов (результат :func:`parse_tmx`).
        threshold: минимальный процент сходства (0–100).
        context_before: предыдущий сегмент документа; если он совпадает с
            ``prev_source`` кандидата, а сходство равно 100 %, тип совпадения —
            ICE.
        top_n: максимальное число возвращаемых кандидатов.

    Returns:
        Список словарей, отсортированный по убыванию ``score``, каждый со
        ключами: ``tuid``, ``source_segment``, ``target_segment``,
        ``score`` (float, 0–100), ``lev_ratio``, ``token_sort_ratio``,
        ``match_type`` (одно из ``"ICE (101%)"``, ``"Exact (100%)"``,
        ``"High Fuzzy"``, ``"Fuzzy"``, ``"Low Fuzzy"``).
        Пустой список, если совпадений выше порога нет.
    """
    # TODO: Студент пишет код здесь
    raise NotImplementedError


def apply_term_glossary(text: str, glossary_df: pd.DataFrame,
                        template: str = "{source} [{target}]") -> str:
    """Разметить исходный текст переводными эквивалентами из глоссария.

    Требования: поиск регистронезависимый, но строго по границам слов; более
    длинные термины обрабатываются раньше коротких; уже подставленные фрагменты
    повторно не размечаются.

    Args:
        text: исходный сегмент.
        glossary_df: таблица глоссария с колонками ``source_term``/``target_term``.
        template: шаблон подстановки с полями ``{source}`` и ``{target}``.

    Returns:
        Текст с размеченными терминами; регистр исходного текста сохраняется.
        Если глоссарий пуст или текст пустой — возвращается исходный текст.
    """
    # TODO: Студент пишет код здесь
    raise NotImplementedError

### 4.1. Вспомогательные функции (реализуйте при необходимости)

Эти функции не оцениваются отдельно, но нужны для `calculate_fuzzy_match`.

In [ ]:
def normalize(text: str) -> str:
    """Нормализовать строку: NFKC, нижний регистр, без пунктуации, одиночные пробелы."""
    # TODO: Студент пишет код здесь
    raise NotImplementedError


def levenshtein_ratio(a: str, b: str) -> float:
    """Процент сходства строк на основе расстояния Левенштейна.

    Формула: ``(1 - distance / max(len(a), len(b))) * 100``.
    Для двух пустых строк вернуть 100.0.
    """
    # TODO: Студент пишет код здесь
    raise NotImplementedError


def token_sort_ratio(a: str, b: str) -> float:
    """Процент сходства при игнорировании порядка слов.

    Нормализовать обе строки, разбить на токены, отсортировать, склеить обратно
    и сравнить через :func:`levenshtein_ratio`.
    """
    # TODO: Студент пишет код здесь
    raise NotImplementedError

## Блок 5. Автотесты

Запустите ячейки ниже **после** реализации функций. Все проверки должны
завершиться без `AssertionError`. Тесты работают на демо-файлах из Блока 3.

In [ ]:
# --- Тест 1: метрики сходства -------------------------------------------
assert abs(levenshtein_ratio("telemetry", "telemetry") - 100.0) < 1e-6, \
    "Одинаковые строки должны давать 100 %"
assert levenshtein_ratio("", "") == 100.0, "Две пустые строки — 100 %"
assert 0.0 <= levenshtein_ratio("payload", "airframe") < 50.0, \
    "Непохожие строки должны давать низкий процент"
assert abs(token_sort_ratio("angle of attack", "attack of angle") - 100.0) < 1e-6, \
    "Token Sort Ratio должен игнорировать порядок слов"
assert normalize("  Angle   OF ATTACK, critical! ") == "angle of attack critical", \
    "normalize: нижний регистр, без пунктуации, одиночные пробелы"
print("Тест 1 (метрики сходства) — OK")

In [ ]:
# --- Тест 2: парсинг TBX -------------------------------------------------
glossary_df = parse_tbx("demo.tbx")

assert isinstance(glossary_df, pd.DataFrame), "parse_tbx должен вернуть DataFrame"
assert len(glossary_df) == 3, f"Ожидается 3 термина, получено {len(glossary_df)}"
for column in ["entry_id", "source_term", "target_term", "pos",
               "definition", "subject_field"]:
    assert column in glossary_df.columns, f"Нет колонки '{column}'"

row = glossary_df.loc[glossary_df["source_term"] == "flight controller"].iloc[0]
assert row["target_term"] == "полётный контроллер", "Неверный русский эквивалент"
assert row["entry_id"] == "tid-001", "Не извлечён атрибут id у termEntry"
assert row["pos"] == "noun", "Не извлечён termNote[@type='partOfSpeech']"
assert "telemetry link" in set(glossary_df["source_term"]), \
    "Термин из <tig> без termNote должен быть извлечён"
print("Тест 2 (парсинг TBX) — OK")
glossary_df.head()

In [ ]:
# --- Тест 3: парсинг TMX -------------------------------------------------
tm_df = parse_tmx("demo.tmx")

assert isinstance(tm_df, pd.DataFrame), "parse_tmx должен вернуть DataFrame"
assert len(tm_df) == 3, f"Ожидается 3 единицы перевода, получено {len(tm_df)}"
for column in ["tuid", "source_segment", "target_segment", "domain",
               "src_len", "prev_source"]:
    assert column in tm_df.columns, f"Нет колонки '{column}'"

first = tm_df.iloc[0]
assert first["source_segment"].startswith("The flight controller"), \
    "Не извлечён текст <seg> исходного языка"
assert first["target_segment"].startswith("Полётный контроллер"), \
    "Не извлечён текст <seg> языка перевода"
assert first["prev_source"] == "", "У первой строки prev_source должен быть пустым"
assert tm_df.iloc[1]["prev_source"] == first["source_segment"], \
    "prev_source должен содержать предыдущий исходный сегмент"
assert first["src_len"] == len(first["source_segment"]), "Неверная длина сегмента"
print("Тест 3 (парсинг TMX) — OK")
tm_df.head()

In [ ]:
# --- Тест 4: Fuzzy Match и ICE ------------------------------------------
exact_query = "The operator checks the telemetry link."
matches = calculate_fuzzy_match(exact_query, tm_df, threshold=70.0)
assert matches, "Точное совпадение обязано быть найдено"
assert abs(matches[0]["score"] - 100.0) < 1e-6, "Точное совпадение — 100 %"
assert matches[0]["match_type"] == "Exact (100%)", \
    "Без контекста 100 % — это Exact, а не ICE"

ice = calculate_fuzzy_match(
    exact_query, tm_df, threshold=70.0,
    context_before="The flight controller stabilises the aircraft.")
assert ice[0]["match_type"] == "ICE (101%)", \
    "При совпадении предыдущего сегмента 100 % должно стать ICE"

fuzzy = calculate_fuzzy_match("The engineer checks the telemetry link.",
                              tm_df, threshold=70.0)
assert fuzzy, "Похожий сегмент должен попадать в выдачу"
assert 75.0 <= fuzzy[0]["score"] < 100.0, \
    f"Ожидается Fuzzy 75–99 %, получено {fuzzy[0]['score']}"
assert fuzzy[0]["match_type"] in {"Fuzzy", "High Fuzzy"}, "Неверный тип совпадения"

scores = [m["score"] for m in calculate_fuzzy_match("telemetry link check",
                                                    tm_df, threshold=0.0)]
assert scores == sorted(scores, reverse=True), \
    "Результаты должны быть отсортированы по убыванию сходства"
assert calculate_fuzzy_match("Совершенно посторонний текст про бухгалтерию.",
                             tm_df, threshold=75.0) == [], \
    "Ниже порога результатов быть не должно"
assert len(calculate_fuzzy_match("telemetry", tm_df, threshold=0.0, top_n=2)) <= 2, \
    "Параметр top_n не учитывается"
print("Тест 4 (Fuzzy Match и ICE) — OK")

In [ ]:
# --- Тест 5: подстановка терминов ---------------------------------------
marked = apply_term_glossary("The flight controller reads the payload status.",
                             glossary_df)
assert "[полётный контроллер]" in marked, "Термин из глоссария не подставлен"
assert "[целевая нагрузка]" in marked, "Подставлен не каждый найденный термин"
assert "flight controller" in marked, "Исходный термин должен сохраняться"

assert apply_term_glossary("No terms here at all.", glossary_df) == \
    "No terms here at all.", "Текст без терминов не должен меняться"

case_test = apply_term_glossary("Payload mass is 2 kg.", glossary_df)
assert "Payload [целевая нагрузка]" in case_test, \
    "Поиск должен быть регистронезависимым, а регистр текста — сохраняться"

assert "[целевая нагрузка]" not in apply_term_glossary("payloads are heavy",
                                                       glossary_df), \
    "Совпадение должно быть по границам слов, а не по подстроке"
print("Тест 5 (подстановка терминов) — OK")
print(marked)

## Блок 6. Работа со своим вариантом

После прохождения всех тестов выполните задание на данных своего варианта.

In [ ]:
VARIANT = 0          # TODO: укажите номер своего варианта
SURNAME = "Ivanov"   # TODO: укажите свою фамилию латиницей

TBX_FILE = f"variant_{VARIANT}.tbx"   # не менее 15 терминов
TMX_FILE = f"variant_{VARIANT}.tmx"   # не менее 10 параллельных предложений

# TODO: 1) создайте файлы своего варианта (генератором или вручную)
# TODO: 2) my_glossary = parse_tbx(TBX_FILE)
# TODO: 3) my_tm = parse_tmx(TMX_FILE)
# TODO: 4) прогоните 3 запроса: ICE, Fuzzy 75-99 %, No Match
# TODO: 5) экспортируйте глоссарий в CSV для Smartcat:
#          columns = ["Source (en)", "Target (ru)", "Comment"],
#          to_csv(..., index=False, encoding="utf-8-sig")
# TODO: 6) выведите итоговую таблицу совпадений

## Блок 7. Выводы

Ответьте письменно (3–5 предложений на пункт):

1. Чем концептуально-ориентированная организация TBX отличается от
   сегментно-ориентированной организации TMX?
2. Почему `xml:lang` требует особой обработки при парсинге и как вы её решили?
3. В каких случаях Levenshtein ratio даёт заниженную оценку сходства и как это
   компенсирует Token Sort Ratio?
4. Чем ICE (101 %) отличается от обычного 100 %-совпадения и почему это важно
   для стоимости проекта в Smartcat?
5. Какие проблемы возникли при автоматической подстановке терминов и как вы их
   решили?

**Ваши выводы:**

_______________________________________________

---

### Что сдать на проверку

1. Ссылка на Colab-ноутбук `lab_01_Фамилия.ipynb` (доступ «для всех по ссылке»).
2. Файлы `variant_NN.tbx`, `variant_NN.tmx`, `glossary_smartcat.csv`.
3. Отчёт с выводами и скриншотом импортированного глоссария в Smartcat.